In [1]:
import io
import zipfile
from pathlib import Path
from urllib.parse import quote

import httpx
from tqdm.auto import tqdm
from minsearch import AppendableIndex
from gitsource import chunk_documents
from toyaikit.chat.interface import IPythonChatInterface
from toyaikit.chat.runners import DisplayingRunnerCallback
from toyaikit.chat.runners import AnthropicMessagesRunner
from toyaikit.llm import AnthropicClient
from toyaikit.tools import Tools

In [2]:
PROMPT_TEMPLATE = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

INSTRUCTIONS = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Make 3 iterations:

1) in the first iteration, perform one search
2) in the second interation, analyze the results from the previous search
   and perform 2 more searches
3) synthesise the results into the output

IMPORTANT: at each step, give an explanation of why you want to perform 
search for this particular search query. It should be 2-3 sentences explaining
the logic of your decision.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Pulumi, so you don't need to 
include the word 'pulumi' in search results
"""

search_tool = {
    "name": "search",
    "description": "Search the documentation database for relevant results based on a query string.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up in the index",
            }
        },
        "required": ["query"],
    },
}

add_entry_tool = {
    "name": "add_entry",
    "description": "Add a new documentation entry to the index.",
    "parameters": {
        "type": "object",
        "properties": {
            "filename": {
                "type": "string",
                "description": "The source filename associated with the entry",
            },
            "content": {
                "type": "string",
                "description": "The full content of the documentation entry",
            },
        },
        "required": ["filename", "content"],
    },
}

In [3]:
def get_or_download_documentation(
    repo_owner: str,
    repo_name: str,
    branch_name: str,
    docs_path: str = "docs",
    save_dir: str | None = None,
) -> bytes:
    """
    Return the Markdown documentation as a zip archive, reusing a local copy if one already exists, and downloading it otherwise.

    Args:
        repo_owner (str): The owner of the GitHub repository.
        repo_name (str): The name of the GitHub repository.
        branch_name (str): The branch to download from.
        docs_path (str): Path of the documentation folder inside the repository.
        save_dir (str | None): Local directory to reuse/save the files in. If
            None, the files are only returned as an in-memory zip.

    Returns:
        bytes: A zip archive containing only the Markdown files under `docs_path`.
    """

    def _get(doc_dir: str | None = None) -> bytes | None:
        """Build a zip from an already-downloaded local directory.

        Args:
            doc_dir (str | None): Directory that may already contain the files.

        Returns:
            bytes | None: A zip archive of the local files, or None if the
            directory is missing or empty.
        """
        if doc_dir is None:
            return None

        out_dir = Path(doc_dir)
        existing_files = sorted(out_dir.rglob("*")) if out_dir.exists() else []
        existing_files = [f for f in existing_files if f.is_file()]
        if not existing_files:
            return None

        print(f"Using already-downloaded documentation in '{doc_dir}' ({len(existing_files)} files)")

        buffer = io.BytesIO()
        with zipfile.ZipFile(buffer, "w") as archive:
            for file in existing_files:
                archive.write(file, file.relative_to(out_dir).as_posix())
        return buffer.getvalue()

    def _download() -> bytes:
        """Download only the Markdown documentation from GitHub.

        Returns:
            bytes: A zip archive containing only the Markdown files under
            `docs_path`.
        """
        api = "https://api.github.com"
        raw = "https://raw.githubusercontent.com"

        with httpx.Client(follow_redirects=True) as client:
            # Resolve the branch to its root tree SHA
            branch = client.get(f"{api}/repos/{repo_owner}/{repo_name}/branches/{branch_name}")
            branch.raise_for_status()
            root_tree_sha = branch.json()["commit"]["commit"]["tree"]["sha"]
            print(f"Retrieved root tree SHA: {root_tree_sha}")

            # Locate the docs subtree inside the root tree
            root_tree = client.get(f"{api}/repos/{repo_owner}/{repo_name}/git/trees/{root_tree_sha}")
            root_tree.raise_for_status()
            docs_entry = next(
                (item for item in root_tree.json()["tree"] if item["path"] == docs_path),
                None,
            )
            if docs_entry is None:
                raise ValueError(f"Path '{docs_path}' not found in {repo_owner}/{repo_name}@{branch_name}")
            print(f"Retrieved docs subtree '{docs_path}' (SHA: {docs_entry['sha']})")

            # Recursively list every file under docs/
            docs_tree = client.get(
                f"{api}/repos/{repo_owner}/{repo_name}/git/trees/{docs_entry['sha']}?recursive=1"
            )
            docs_tree.raise_for_status()
            print(f"Retrieved recursive tree for '{docs_path}'")

            # Collect the Markdown files to download
            markdown_files = [
                item["path"]
                for item in docs_tree.json().get("tree", [])
                if item.get("type") == "blob" and item["path"].endswith((".md", ".mdx"))
            ]
            print(f"Found {len(markdown_files)} Markdown files to download")

            # Download each Markdown file and package it into an in-memory zip
            buffer = io.BytesIO()
            with zipfile.ZipFile(buffer, "w") as archive:
                for path in tqdm(markdown_files, desc="Downloading docs", unit="file"):
                    file_url = f"{raw}/{repo_owner}/{repo_name}/{branch_name}/{quote(f'{docs_path}/{path}')}"
                    response = client.get(file_url)
                    response.raise_for_status()
                    archive.writestr(path, response.content)

        return buffer.getvalue()

    # Reuse the local copy if it already exists
    data = _get(save_dir)
    if data is not None:
        return data

    # Otherwise download and (optionally) save to disk
    data = _download()

    if save_dir is not None:
        out_dir = Path(save_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(io.BytesIO(data)) as zf:
            num_files = len(zf.namelist())
            zf.extractall(out_dir)
        print(f"Extracted {num_files} files to '{save_dir}'")

    return data

In [4]:
def retrieve_documentation(
    repo_owner: str,
    repo_name: str,
    branch_name: str,
    docs_path: str = "docs",
    save_dir: str | None = None,
):
    response_contents = get_or_download_documentation(
        repo_owner=repo_owner,
        repo_name=repo_name,
        branch_name=branch_name,
        docs_path=docs_path,
        save_dir=save_dir,
    )
    documents = []
    with zipfile.ZipFile(io.BytesIO(response_contents)) as zip_ref:
        for file_path in zip_ref.namelist():
            with zip_ref.open(file_path) as file:
                content = file.read().decode('utf-8')
                doc = {
                    'content': content,
                    'filename': file_path
                }
                documents.append(doc)
    return documents

In [5]:
documents = retrieve_documentation(
    repo_owner="pulumi",
    repo_name="docs",
    branch_name="master",
    docs_path="content",
    save_dir="docs",
)
documents[:5]

Using already-downloaded documentation in 'docs' (2186 files)


[{'content': '---\ntitle: The infrastructure as code platform for the AI era – Pulumi\nmeta_desc: Pulumi is an open-source infrastructure as code platform that helps humans and agents build and manage cloud infrastructure with real programming languages.\nmeta_image: /images/home/home-meta.png\n\ninclude_organization_schema: true\n\nsections:\n  - type: hero\n    layout: split\n    cta_secondary_text: Download open source\n    cta_secondary_link: /docs/install/\n    badge_highlight_text: "New:"\n    badge_text: "Native support for Terraform and HCL"\n    badge_link: /releases/terraform-state-backend-modules-hcl/\n    title: "*Unleash agents* <br>on your infrastructure."\n    description: |\n      Ship infrastructure faster with tools that let agents do what they do best — without letting them go off the rails.\n    anchor: hero\n    hero_animation: agent-loop\n\n  - type: logo_grid\n    title: Trusted by over 4,000 innovative companies\n    logos:\n      - snowflake\n      - nvidia\n  

In [6]:
document_chunks = chunk_documents(documents, size=3000, step=1500)

In [7]:
index = AppendableIndex(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(document_chunks)

In [8]:
def add_entry(filename: str, content: str):
    entry = {
        "start": 0,
        "content": content,
        "filename": filename,
    }
    index.append(entry)
    return "Entry added"

def search(query: str, top_k: int = 5):
    return index.search(query, num_results=top_k)

In [9]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)
agent_tools.add_tool(add_entry, add_entry_tool)

In [10]:
agent_tools.get_tools()

[{'name': 'search',
  'description': 'Search the documentation database for relevant results based on a query string.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'The search query to look up in the index'}},
   'required': ['query']}},
 {'name': 'add_entry',
  'description': 'Add a new documentation entry to the index.',
  'parameters': {'type': 'object',
   'properties': {'filename': {'type': 'string',
     'description': 'The source filename associated with the entry'},
    'content': {'type': 'string',
     'description': 'The full content of the documentation entry'}},
   'required': ['filename', 'content']}}]

In [11]:
llm_client = AnthropicClient(model="claude-opus-5")
agent = AnthropicMessagesRunner(
    tools=agent_tools,
    developer_prompt=INSTRUCTIONS,
    # chat_interface=,
    llm_client=llm_client,
)

In [12]:
result = agent.loop("How do I launch an EC2 in a private net in AWS?")

In [13]:
print(result.last_message)

## Iteration 3: Synthesis

Here's how to launch an EC2 instance in a private network on AWS with Pulumi, based on what's in the knowledge base.

---

### Step 1: Create the private network

You have two documented options.

**Option A — `awsx.ec2.Vpc` (the Crosswalk component).** Per `docs/iac/concepts/components/_index.md`, this component "creates a complete VPC—including subnets, route tables, and internet gateways—behind a simple interface." You declare subnet tiers with `subnetSpecs` and it exposes `privateSubnetIds`:

```typescript
import * as awsx from "@pulumi/awsx";

const vpc = new awsx.ec2.Vpc("vpc", {
    subnetSpecs: [
        { type: awsx.ec2.SubnetType.Public,  cidrMask: 22 },
        { type: awsx.ec2.SubnetType.Private, cidrMask: 20 },
    ],
}, { protect: true });

export const vpcId = vpc.vpcId;
export const privateSubnetIds = vpc.privateSubnetIds;
export const publicSubnetIds = vpc.publicSubnetIds;
```

```python
import pulumi, pulumi_awsx as awsx

vpc = awsx.ec2.Vpc(

In [14]:
float(result.cost.total_cost) if result.cost else 0.00

0.25882

In [15]:
chat_interface = IPythonChatInterface()
runner_callback = DisplayingRunnerCallback(chat_interface=chat_interface)  # type: ignore

In [16]:
result = agent.loop(
    "How do I launch an EC2 in a private net in AWS?",
    callback=runner_callback,
)

-> Response received


-> Response received


-> Response received


In [17]:
result2 = agent.loop(
    "Show me how to do that in python",
    callback=runner_callback,
    previous_messages=result.all_messages,
)

-> Response received


In [18]:
result3 = agent.loop(
    "Show me the code in typescript",
    callback=runner_callback,
    previous_messages=result.all_messages,
)

-> Response received


In [19]:
agent = AnthropicMessagesRunner(
    tools=agent_tools,
    developer_prompt=INSTRUCTIONS,
    chat_interface=chat_interface,  # type: ignore
    llm_client=llm_client,
)

In [20]:
result = agent.run()

Chat ended.


In [21]:
float(result.cost.total_cost) if result.cost else 0.00

0.0